# 05 · ResNet experts (Pool B)

Trains **one ResNet per generator** (real vs. that generator's fakes), calibrates each one, then scores
every image with every expert. The output is the ResNet half of the **score table** that all combiners use.

```
cleaned images + manifest.csv
        │
        ├─► 1. train 7 experts   (expert_train → fit, expert_val → pick best epoch)
        ├─► 2. calibrate          (temperature scaling on expert_val)
        ├─► 3. score              (every expert × every image in meta_train / meta_val / test / seen_test)
        └─► 4. sanity check       (expert × generator AUROC matrix on meta_val)
```

**Rules this notebook enforces**
- An expert only ever trains on `expert_train` rows of **its own** generator (its reals + its fakes).
- `expert_val` is used only to pick the epoch and fit the temperature.
- `meta_*`, `test` and `seen_test` images are **never** seen during expert training (checked in step 2).
- The `test` role is scored but never looked at here; the sanity matrix uses `meta_val`.

**Where to run:** Kaggle with **GPU (T4/P100)** and **Internet ON** (for the ImageNet weights), or locally.
Expected time on a T4 with ResNet-50: ~10–20 min per expert + ~30 min scoring.
Set `SMOKE_TEST = True` first: it runs the whole notebook on a tiny subset in a few minutes.

**Resumable:** finished experts and finished scoring parts are saved and skipped on re-run,
so a crashed or timed-out session just continues where it stopped.

## 0 · Config — the only cell you normally edit

In [ ]:
from pathlib import Path
import os

ON_KAGGLE = os.path.exists("/kaggle/input")

# --- Paths -------------------------------------------------------------------
# IMG_ROOT : folder with the cleaned images (output of 02_clean_subset)
# MANIFEST : CSV with one row per image; columns: path, generator, label, role
#            path      = relative to IMG_ROOT (or absolute)
#            generator = one of GENERATORS below
#            label     = 0 real / 1 fake   (also accepts "real"/"nature"/"fake"/"ai")
#            role      = expert_train | expert_val | meta_train | meta_val | test | seen_test
#            If MANIFEST does not exist, it is built from folders:
#            IMG_ROOT/<generator>/<role>/<real|nature|fake|ai>/*.jpg
if ON_KAGGLE:
    IMG_ROOT = Path("/kaggle/input/genimage-clean")      # name of your cleaned dataset
    OUT_DIR  = Path("/kaggle/working/resnet_experts")
else:
    IMG_ROOT = Path("data/clean")
    OUT_DIR  = Path("outputs/resnet_experts")
MANIFEST = IMG_ROOT / "manifest.csv"

# --- Experiment ----------------------------------------------------------------
GENERATORS = ["adm", "biggan", "glide", "midjourney", "sd", "vqdm", "wukong"]  # SD 1.4 + 1.5 merged as "sd"
ARCH         = "resnet50"   # "resnet18" if GPU time is short
PRETRAINED   = True         # ImageNet weights (Kaggle: Settings → Internet ON)
WEIGHTS_FILE = None         # optional local .pth with ImageNet weights (used instead of downloading)
IMG_SIZE     = 224          # crop size; images are cropped, never resized, if they are >= this size
EPOCHS       = 8            # max epochs per expert
PATIENCE     = 2            # stop if val loss does not improve for this many epochs
BATCH_SIZE   = 64           # lower to 32/16 if you run out of memory (e.g. on a Mac)
LR           = 1e-4
WEIGHT_DECAY = 1e-4
TEMP_BOUNDS  = (0.5, 50.0)  # allowed range for the calibration temperature (see step 4)
SEED         = 0
SCORE_ROLES  = ["expert_val", "meta_train", "meta_val", "test", "seen_test"]

# --- Quick check ------------------------------------------------------------------
SMOKE_TEST = False          # True = tiny run (SMOKE_N images per generator/role/class, 1 epoch)
SMOKE_N    = 40

## 1 · Setup

In [ ]:
import json, time, random, platform, glob, shutil
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models as tvm
from PIL import Image
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")            # Apple M-chip
    return torch.device("cpu")

DEVICE   = pick_device()
USE_AMP  = DEVICE.type == "cuda"              # fp16 mixed precision only on NVIDIA GPUs
# Worker processes speed up image loading. In Jupyter on macOS/Windows they break with classes
# defined in the notebook, so we only use them on Linux (Kaggle).
NUM_WORKERS = min(4, os.cpu_count() or 1) if platform.system() == "Linux" else 0

if SMOKE_TEST:
    EPOCHS, PATIENCE = 1, 1
    OUT_DIR = OUT_DIR.parent / (OUT_DIR.name + "_smoke")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
torch.backends.cudnn.benchmark = True        # faster; tiny run-to-run differences are expected

CKPT_DIR, PART_DIR = OUT_DIR / "checkpoints", OUT_DIR / "score_parts"
for d in (OUT_DIR, CKPT_DIR, PART_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"device={DEVICE}  amp={USE_AMP}  workers={NUM_WORKERS}  torch={torch.__version__}  "
      f"torchvision={torchvision.__version__}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("OUT_DIR:", OUT_DIR.resolve())

## 2 · Load the manifest and check it

The checks stop the notebook early if the split is broken: wrong generator names, an image in two roles,
or a missing class.

In [ ]:
def find_manifest():
    """Use MANIFEST if it exists; on Kaggle also search the attached inputs for manifest.csv."""
    if MANIFEST.exists():
        return MANIFEST
    if ON_KAGGLE:
        hits = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
        if hits:
            print("Found manifest at", hits[0])
            return Path(hits[0])
    return None

LABEL_MAP = {"0": 0, "1": 1, "real": 0, "nature": 0, "fake": 1, "ai": 1}

def build_manifest_from_folders(root):
    """Fallback: IMG_ROOT/<generator>/<role>/<real|nature|fake|ai>/<image>."""
    rows = []
    for p in root.rglob("*"):
        if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        parts = p.relative_to(root).parts
        if len(parts) < 4:
            continue
        gen, role, cls = parts[0], parts[1], parts[2]
        rows.append({"path": str(p.relative_to(root)), "generator": gen, "role": role,
                     "label": LABEL_MAP.get(cls.lower())})
    return pd.DataFrame(rows)

mpath = find_manifest()
if mpath is not None:
    df = pd.read_csv(mpath)
    IMG_ROOT = mpath.parent if not (IMG_ROOT / str(df["path"].iloc[0])).exists() else IMG_ROOT
else:
    print("No manifest.csv found, building it from the folder layout ...")
    df = build_manifest_from_folders(IMG_ROOT)

# --- normalise columns ------------------------------------------------------------
missing = {"path", "generator", "label", "role"} - set(df.columns)
assert not missing, f"manifest is missing columns: {missing}"
df["generator"] = df["generator"].astype(str).str.lower()
df["role"] = df["role"].astype(str).str.lower()
df["label"] = df["label"].astype(str).str.lower().map(LABEL_MAP)
assert df["label"].notna().all(), "some labels are not 0/1/real/fake"
df["label"] = df["label"].astype(int)
df["abs_path"] = [p if os.path.isabs(p) else str(IMG_ROOT / p) for p in df["path"].astype(str)]
if "image_id" not in df.columns:
    df["image_id"] = df["path"].astype(str)

# --- checks -------------------------------------------------------------------------
unknown = sorted(set(df["generator"]) - set(GENERATORS))
assert not unknown, f"generators not in GENERATORS: {unknown}"
dupes = df[df.duplicated("abs_path", keep=False)]
assert dupes.empty, f"{len(dupes)} images appear in more than one row/role, e.g. {dupes['path'].iloc[0]}"
for g in GENERATORS:
    for role in ("expert_train", "expert_val"):
        n = df[(df.generator == g) & (df.role == role)]["label"].value_counts()
        assert n.get(0, 0) > 0 and n.get(1, 0) > 0, f"{g}/{role} needs both real and fake images: {dict(n)}"
missing_files = [p for p in df["abs_path"].sample(min(200, len(df)), random_state=0) if not os.path.exists(p)]
assert not missing_files, f"files not found, e.g. {missing_files[:3]}  (check IMG_ROOT)"

if SMOKE_TEST:  # keep SMOKE_N images per generator / role / class
    df = (df.sample(frac=1, random_state=SEED)
            .groupby(["generator", "role", "label"]).head(SMOKE_N)
            .reset_index(drop=True))

counts = df.pivot_table(index=["generator"], columns=["role", "label"], values="path",
                        aggfunc="count", fill_value=0)
print(f"{len(df):,} images  |  roles: {sorted(df.role.unique())}")
counts

## 3 · Data pipeline and model

- **Train:** random 224 crop + horizontal flip. **Eval:** centre 224 crop.
- Images are **cropped, not resized** (resizing would blur the very pixel traces the expert should learn).
  Only images smaller than 224 px are upscaled, and the cleaning step should already prevent that.
- ImageNet normalisation, because the backbone was pretrained on ImageNet.
- Model: ImageNet-pretrained ResNet with the 1000-class head replaced by **one output (logit: >0 = fake)**.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class UpscaleIfSmaller:
    """Only upscale images whose short side is below `size`; never touch larger ones."""
    def __init__(self, size): self.size = size
    def __call__(self, img):
        return T.functional.resize(img, self.size) if min(img.size) < self.size else img

train_tf = T.Compose([UpscaleIfSmaller(IMG_SIZE), T.RandomCrop(IMG_SIZE), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf  = T.Compose([UpscaleIfSmaller(IMG_SIZE), T.CenterCrop(IMG_SIZE),
                      T.ToTensor(), T.Normalize(MEAN, STD)])

class ImageDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths  = frame["abs_path"].tolist()
        self.labels = frame["label"].to_numpy(np.float32)
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.tf(im.convert("RGB"))
        return x, self.labels[i]

def make_loader(frame, train):
    return DataLoader(ImageDataset(frame, train_tf if train else eval_tf),
                      batch_size=BATCH_SIZE if train else BATCH_SIZE * 2,
                      shuffle=train, drop_last=train and len(frame) > BATCH_SIZE,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)

def build_model():
    ctor, weights = {"resnet18": (tvm.resnet18, tvm.ResNet18_Weights.IMAGENET1K_V1),
                     "resnet50": (tvm.resnet50, tvm.ResNet50_Weights.IMAGENET1K_V2)}[ARCH]
    model = ctor(weights=weights if (PRETRAINED and WEIGHTS_FILE is None) else None)
    if WEIGHTS_FILE:
        state = torch.load(WEIGHTS_FILE, map_location="cpu")
        state = {k: v for k, v in state.items() if not k.startswith("fc.")}
        res = model.load_state_dict(state, strict=False)
        assert not res.unexpected_keys and set(res.missing_keys) <= {"fc.weight", "fc.bias"}, res
    model.fc = nn.Linear(model.fc.in_features, 1)          # one logit: real (<0) vs fake (>0)
    return model.to(DEVICE).to(memory_format=torch.channels_last)

@torch.no_grad()
def predict_logits(model, loader):
    """Returns (logits, labels) as float32 numpy arrays."""
    model.eval()
    out, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out.append(model(x).float().squeeze(1).cpu())
        ys.append(y)
    return torch.cat(out).numpy(), torch.cat(ys).numpy()

def metrics(logits, labels):
    lt, yt = torch.from_numpy(logits), torch.from_numpy(labels)
    return {"loss": F.binary_cross_entropy_with_logits(lt, yt).item(),
            "acc":  float(((logits > 0) == (labels > 0.5)).mean()),
            "auc":  float(roc_auc_score(labels, logits)) if len(set(labels)) == 2 else float("nan")}

print(f"{ARCH}: {sum(p.numel() for p in build_model().parameters())/1e6:.1f} M parameters")

## 4 · Train one expert per generator

For each generator: fine-tune the whole network on its `expert_train` images, check `expert_val` after every
epoch, keep the epoch with the **lowest validation loss** (accuracy and AUROC quickly hit ~100% and stop
being informative), and stop early after `PATIENCE` epochs without improvement.

**Calibration (temperature scaling).** We fit one number *T* per expert on `expert_val` so that
`sigmoid(logit / T)` is a well-calibrated probability. If validation is perfectly separated, the best *T*
is "infinitely sharp", which would make the expert overconfident on other generators; `TEMP_BOUNDS` keeps
*T* in a sensible range. Raw logits are kept as well, so combiners can recalibrate.

In [ ]:
def fit_temperature(logits, labels):
    """Temperature scaling: minimise validation NLL of sigmoid(logit / T) over T."""
    lt, yt = torch.from_numpy(logits).double(), torch.from_numpy(labels).double()
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(lt / log_t.exp(), yt)
        loss.backward()
        return loss
    opt.step(closure)
    t_raw = log_t.detach().exp().item()
    t = float(np.clip(t_raw, *TEMP_BOUNDS))
    if t != t_raw:
        print(f"    temperature {t_raw:.3g} clipped to {t:.3g}")
    return t

def train_expert(gen):
    ckpt_path = CKPT_DIR / f"{gen}.pt"
    if ckpt_path.exists():
        print(f"[{gen}] already trained, skipping (delete {ckpt_path.name} to retrain)")
        return torch.load(ckpt_path, map_location="cpu", weights_only=False)["info"]

    seed_everything(SEED)
    tr = df[(df.generator == gen) & (df.role == "expert_train")]
    va = df[(df.generator == gen) & (df.role == "expert_val")]
    train_loader, val_loader = make_loader(tr, train=True), make_loader(va, train=False)

    model = build_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = EPOCHS * len(train_loader)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best, best_state, bad, history = float("inf"), None, 0, []
    t0 = time.time()
    print(f"[{gen}] train {len(tr):,} (fake {tr.label.mean():.0%}) | val {len(va):,}")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        run_loss, n = 0.0, 0
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = F.binary_cross_entropy_with_logits(model(x).squeeze(1).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item() * len(y); n += len(y)

        val_logits, val_y = predict_logits(model, val_loader)
        m = metrics(val_logits, val_y)
        history.append({"epoch": epoch, "train_loss": run_loss / n, **{f"val_{k}": v for k, v in m.items()}})
        print(f"  epoch {epoch}: train_loss {run_loss/n:.4f} | val_loss {m['loss']:.4f} "
              f"acc {m['acc']:.4f} auc {m['auc']:.4f} | {time.time()-t0:.0f}s")
        if m["loss"] < best - 1e-5:
            best, bad = m["loss"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stop"); break

    model.load_state_dict(best_state)
    val_logits, val_y = predict_logits(model, val_loader)
    temp = fit_temperature(val_logits, val_y)
    info = {"generator": gen, "arch": ARCH, "temperature": temp, "epochs_run": len(history),
            "best_val": metrics(val_logits, val_y), "train_seconds": round(time.time() - t0),
            "n_train": len(tr), "n_val": len(va), "history": history}
    torch.save({"state_dict": best_state, "info": info}, ckpt_path)
    print(f"[{gen}] done: val auc {info['best_val']['auc']:.4f}, T={temp:.3f}, {info['train_seconds']}s\n")
    del model, opt; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    return info

# Leakage guard: nothing an expert trains on may appear in the roles we score.
train_paths = set(df.loc[df.role == "expert_train", "abs_path"])
score_paths = set(df.loc[df.role.isin(SCORE_ROLES), "abs_path"])
assert not (train_paths & score_paths), "expert_train images overlap with scored roles!"

expert_info = {g: train_expert(g) for g in GENERATORS}

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
for g, info in expert_info.items():
    h = pd.DataFrame(info["history"])
    axes[0].plot(h.epoch, h.train_loss, marker="o", lw=1.5, ms=4, label=g)
    axes[1].plot(h.epoch, h.val_loss, marker="o", lw=1.5, ms=4, label=g)
for ax, t in zip(axes, ["train loss", "validation loss (expert_val)"]):
    ax.set_title(t, loc="left"); ax.set_xlabel("epoch"); ax.set_ylabel("BCE loss"); ax.grid(alpha=.25)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.spines[["top", "right"]].set_visible(False)
axes[1].legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig(OUT_DIR / "training_curves.png", dpi=150); plt.show()

summary = pd.DataFrame([{"expert": g, "epochs": i["epochs_run"], "val_auc": i["best_val"]["auc"],
                         "val_acc": i["best_val"]["acc"], "val_loss": i["best_val"]["loss"],
                         "temperature": i["temperature"], "minutes": i["train_seconds"] / 60}
                        for g, i in expert_info.items()]).round(4)
summary

## 5 · Score every image with every expert

All 7 experts are loaded once and each image is read once. Scoring is saved **per image-generator**
(`score_parts/<generator>.csv`), so a re-run skips finished parts.

Output columns per expert: `rn_<expert>_logit` (raw) and `rn_<expert>_prob` (= sigmoid(logit / T)).
The held-out expert is **not** removed here; the combiner notebook drops it per fold.

In [ ]:
def load_experts():
    experts = {}
    for g in GENERATORS:
        ck = torch.load(CKPT_DIR / f"{g}.pt", map_location="cpu", weights_only=False)
        m = build_model(); m.load_state_dict(ck["state_dict"]); m.eval()
        if USE_AMP:
            m = m.half()
        experts[g] = (m, ck["info"]["temperature"])
    return experts

@torch.no_grad()
def score_frame(frame, experts):
    loader = make_loader(frame, train=False)
    logits = {g: [] for g in experts}
    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        if USE_AMP:
            x = x.half()
        for g, (m, _) in experts.items():
            logits[g].append(m(x).float().squeeze(1).cpu())
    out = frame[["image_id", "path", "generator", "label", "role"]].reset_index(drop=True).copy()
    for g, (_, temp) in experts.items():
        z = torch.cat(logits[g]).numpy()
        out[f"rn_{g}_logit"] = z
        out[f"rn_{g}_prob"] = 1 / (1 + np.exp(-z / temp))
    return out

to_score = df[df.role.isin(SCORE_ROLES)]
print(f"scoring {len(to_score):,} images with {len(GENERATORS)} experts")
experts = None
for g in GENERATORS:
    part = PART_DIR / f"{g}.csv"
    if part.exists():
        print(f"  {g}: already scored"); continue
    experts = experts or load_experts()
    t0 = time.time()
    score_frame(to_score[to_score.generator == g], experts).to_csv(part, index=False)
    print(f"  {g}: {(to_score.generator == g).sum():,} images in {time.time()-t0:.0f}s")

scores = pd.concat([pd.read_csv(PART_DIR / f"{g}.csv") for g in GENERATORS], ignore_index=True)
try:
    scores.to_parquet(OUT_DIR / "resnet_scores.parquet", index=False)
    print("saved resnet_scores.parquet")
except Exception:             # pyarrow missing → CSV only
    print("parquet not available (pip install pyarrow); CSV is saved instead")
scores.to_csv(OUT_DIR / "resnet_scores.csv.gz", index=False)
print("saved resnet_scores.csv.gz", scores.shape)
scores.head()

## 6 · Sanity check: who detects whom?

AUROC of every expert (rows) on every generator's `meta_val` images (columns). `test` is not used here.

What to expect, from GenImage's own ResNet-50 results:
- **Diagonal ≈ 1.0**: each expert detects its own generator.
- **SD ↔ Wukong high, ADM → GLIDE/BigGAN fairly high**: these are the "sibling" pairs.
- **Most other cells much lower** (often 0.5–0.7): ResNet experts are specialists.

If the diagonal is low, training went wrong. If *everything* is ~1.0, suspect a shortcut
(size or JPEG traces) and rerun the shortcut check.

In [ ]:
val = scores[scores.role == "meta_val"]
check_role = "meta_val"
if val.empty:                       # e.g. smoke test without meta_val rows
    val, check_role = scores[scores.role == "expert_val"], "expert_val (fallback)"
mat = pd.DataFrame(index=GENERATORS, columns=GENERATORS, dtype=float)
for e in GENERATORS:
    for g in GENERATORS:
        sub = val[val.generator == g]
        if sub.label.nunique() == 2:
            mat.loc[e, g] = roc_auc_score(sub.label, sub[f"rn_{e}_logit"])
mat.to_csv(OUT_DIR / "cross_generator_auroc.csv")

fig, ax = plt.subplots(figsize=(6.6, 5.6))
im = ax.imshow(mat.values.astype(float), cmap="Blues", vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(GENERATORS)), GENERATORS, rotation=40, ha="right")
ax.set_yticks(range(len(GENERATORS)), GENERATORS)
ax.set_xlabel("images from generator"); ax.set_ylabel("expert trained on")
ax.set_title(f"ResNet experts: AUROC on {check_role}", loc="left", fontsize=11)
for i in range(len(GENERATORS)):
    for j in range(len(GENERATORS)):
        v = mat.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 0.82 else "#1f2933")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="AUROC (0.5 = chance)")
plt.tight_layout(); plt.savefig(OUT_DIR / "cross_generator_auroc.png", dpi=150); plt.show()
mat.round(3)

## 7 · Save run info

In [ ]:
run_info = {
    "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in {
        "ARCH": ARCH, "PRETRAINED": PRETRAINED, "WEIGHTS_FILE": WEIGHTS_FILE, "IMG_SIZE": IMG_SIZE,
        "EPOCHS": EPOCHS, "PATIENCE": PATIENCE, "BATCH_SIZE": BATCH_SIZE, "LR": LR,
        "WEIGHT_DECAY": WEIGHT_DECAY, "TEMP_BOUNDS": TEMP_BOUNDS, "SEED": SEED,
        "SCORE_ROLES": SCORE_ROLES, "SMOKE_TEST": SMOKE_TEST, "GENERATORS": GENERATORS}.items()},
    "environment": {"torch": torch.__version__, "torchvision": torchvision.__version__,
                    "device": str(DEVICE),
                    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None},
    "experts": expert_info,
    "n_scored": int(len(scores)),
}
(OUT_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2, default=str))
print("Outputs in", OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob("*")):
    size = sum(f.stat().st_size for f in p.rglob("*")) if p.is_dir() else p.stat().st_size
    print(f"  {p.name:32s} {size/1e6:8.1f} MB")

## Next steps

1. **On Kaggle:** *Save Version → Save & Run All*. When it finishes, open the **Output** tab and click
   **New Dataset** (e.g. `resnet-experts`), so the checkpoints and scores are stored safely.
2. **To your laptop:** you only need the scores (a few tens of MB):
   `kaggle datasets download -d <you>/resnet-experts -f resnet_experts/resnet_scores.parquet -p results/`
3. **Merge** with the CLIP expert scores on `image_id` → the full score table for the combiners.
4. **For the report:** `cross_generator_auroc.png`, `training_curves.png`, and the summary table above.

**Known caveat to mention in the report:** GenImage's real images come from ImageNet, which the ResNet
backbone saw during pretraining. All roles draw their reals the same way, so the comparison stays fair,
but absolute numbers may be slightly optimistic.